# Step 1: Prepare features

Ce notebook prépare les attributs au cas par cas:
- traite la connectivité
- complète l'attribut vitesse OSM
- merge les fichiers de source différente: VD, FR, GE pour obtenir 1 fichier GG

In [1]:
import importlib
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

import network_connectivity
import network_slope
importlib.reload(network_connectivity)
importlib.reload(network_slope)

from network_connectivity import *
from network_slope import *



## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters")
territory = 'GG' # GG or GE or OTC (need adaptation in attribute table, for later)
network = "bike"  # walk or bike


input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-1"
include = f"include_in_index_{territory}"

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

Set parameters
Loading segments...


In [3]:
# Pour OTC : créer colonne infra_bike depuis TYPE_AMENA (équivalent de la classification OSM Step 0)
if territory == 'OTC':
    TYPE_AMENA_TO_INFRA_BIKE = {
        'Piste':          'piste_cyclable',
        'Bande':          'bande_cyclable',
        'Voie Bus':       'voie_bus',
        'Dérogation 2R':  'sur_chaussee',
        'Sas':            'sas_velo',
        'Contresens':     'contresens',
    }
    if 'TYPE_AMENA' in segmented_net.columns:
        segmented_net['infra_bike'] = segmented_net['TYPE_AMENA'].map(TYPE_AMENA_TO_INFRA_BIKE).fillna('inconnu')
    else:
        segmented_net['infra_bike'] = 'inconnu'
    print("OTC — infra_bike créé depuis TYPE_AMENA :")
    print(segmented_net['infra_bike'].value_counts())

#### Import des attributs

In [4]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}_WIP.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info.loc[~attributs_info[include].isin(["False", "Unavailable"])].copy()  # Filter out rows where include_in_index is False or 'Unavailable'

attributs_info

,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,socle,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Unnamed: 37,commentaire,extend_m_GG,compute_step2_GG
0,attractivite,amenite,amenite,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,False,0.5,75,75.0,favorable,point,count,NaN,point,count,NaN,4.0,4.0,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
1,attractivite,connectivite,connectivite,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.0,False,0.0,10,10.0,favorable,line,sum,conn_branching_in_buffer,line,sum,conn_branching_in_buffer,7.0,7.0,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
2,attractivite,pente,pente,True,True,opendata,attributs/GE/topographie,pente_2056.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/topographie,pente_2056.tif,prefer_opendata,0.5,False,0.5,10,10.0,defavorable,line,sum,slope_pct_abs_smoothed,line,sum,slope_pct_abs_smoothed,75.0,75.0,keep_nan,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,top,NaN,NaN
3,confort,lac_cours_deau,eau,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,False,0.5,50,50.0,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
4,confort,temperature,temperature,True,True,opendata,attributs/GE/temperature,GrandGeneve_Landsat8_9_SummerMean_LST_2025.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,GrandGeneve_Landsat8_9_SummerMean_LST_2025.tif,prefer_opendata,0.0,False,0.0,250,250.0,defavorable,point,raster,temperature,point,raster,temperature,40.0,40.0,keep_nan,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
5,confort,air,air,True,True,opendata,attributs/GE/air,NO2_Extraction_2025.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/air,NO2_Extraction_2025.tif,prefer_opendata,0.0,False,0.0,750,750.0,defavorable,point,raster,no_2,point,raster,no_2,NaN,NaN,keep_nan,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
6,confort,landuse_env,alentours,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.0,False,0.0,50,50.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
7,confort,canopee,canopee,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/canopee,SIPV_MN_CARTO_GG.gpkg,prefer_opendata,0.5,False,0.5,25,25.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,"GG : couverture du sol SITG, forêts et végétat...",NaN,NaN
8,confort,environnement_ouvert_naturel,environnement_ouvert_naturel,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/environnement_ouvert_naturel,SIPV_MN_CARTO_GG.gpkg,NaN,0.5,False,0.5,30,NaN,favorable,NaN,NaN,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,"GG : couverture du sol SITG, prairies/cultures...",NaN,NaN
9,equipement,stationnement_velo,stationnement_velo,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,False,0.5,30,30.0,favorable,point,sum,score,point,max,score,NaN,5.0,zero_if_missing,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN

## Connectivité du réseau

In [5]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

Data initialized
Processing attribute: connectivite
Index(['u', 'v', 'key', 'infra_bike', 'bike_direction', 'source', 'highway',
       'cycleway', 'cycleway:left', 'cycleway:right', 'cycleway:both',
       'bicycle', 'segregated', 'oneway', 'oneway:bicycle', 'name', 'maxspeed',
       'surface', 'len_m', 'edge_id', 'infra_schema', 'infra_schema_share',
       'schema_piste_ignoree', 'infra_cyclable', 'infra_cyclable_source',
       'length', 'geometry', 'segment_id'],
      dtype='object')


,u,v,key,infra_bike,bike_direction,source,highway,cycleway,cycleway:left,cycleway:right,cycleway:both,bicycle,segregated,oneway,oneway:bicycle,name,maxspeed,surface,len_m,edge_id,infra_schema,infra_schema_share,schema_piste_ignoree,infra_cyclable,infra_cyclable_source,length,geometry,segment_id,length_m,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_intersections_in_buffer,conn_branching_in_buffer,conn_index_score,filtered
0,-9223359204931565246,-7580527903985510507,000000,bande_cyclable,both,OSM_highway_bike,residential,None,None,lane,None,None,None,None,None,Rue du 31-Décembre,None,asphalt,33.141014,0,None,0.0,False,bande_cyclable,osm,33.141014,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",000000,33.141014,False,True,9,5,0.304545,0.304545,1
1,-7580527903985510507,-9223359204931565246,000001,bande_cyclable,both,OSM_highway_bike,residential,None,None,lane,None,None,None,None,None,Rue du 31-Décembre,None,asphalt,33.141014,0,None,0.0,False,bande_cyclable,osm,33.141014,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",000001,33.141014,False,True,9,5,0.304545,0.304545,1
2,1073075817943591750,-7536763922566523035,000002,bike_road,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,None,228.789498,1,None,0.0,False,bike_road,osm,50.000000,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",000002,50.000000,False,True,7,3,0.202273,0.202273,1
3,-7536763922566523035,-81842240814747506,000003,bike_road,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,None,228.789498,1,None,0.0,False,bike_road,osm,50.000000,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",000003,50.000000,False,False,5,0,0.100000,0.100000,1
4,-81842240814747506,5614709667465158974,000004,bike_road,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,None,228.789498,1,None,0.0,False,bike_road,osm,50.000000,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",000004,50.000000,False,False,6,0,0.100000,0.100000,1


In [6]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]], attribute)

Geometries cleaned
Filtered data saved for attribute: connectivite in format: parquet
Filtered data saved for attribute: connectivite in format: csv
Filtered data saved for attribute: connectivite in format: gpkg


In [7]:
segmented_net_index.to_crs(target_crs)[["key","segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]]

,key,segment_id,geometry,conn_branching_in_buffer,conn_index_score,filtered
0,000000,000000,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",0.304545,0.304545,1
1,000001,000001,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",0.304545,0.304545,1
2,000002,000002,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",0.202273,0.202273,1
3,000003,000003,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",0.100000,0.100000,1
4,000004,000004,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",0.100000,0.100000,1
...,...,...,...,...,...,...
620042,620042,620042,"LINESTRING (6.12303 46.18315, 6.12293 46.18314)",0.100000,0.100000,1
620043,620043,620043,"LINESTRING (6.29735 46.45431, 6.29756 46.45439...",0.263636,0.263636,1
620044,620044,620044,"LINESTRING (6.2973 46.4544, 6.29727 46.45439)",0.263636,0.263636,1
620045,620045,620045,"LINESTRING (6.29727 46.45439, 6.29736 46.45442...",0.263636,0.263636,1


## Pente

In [8]:
# Initialize
attribute = None
row = None
segmented_net_slope = None
print("Data initialized")

# Load
###----change here-----
attribute = 'pente'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

source_path_col = f"source_path_{territory}"
file_name_col = f"file_name_{territory}"
raster_path = f"{input_file_path}/{row[source_path_col]}/{row[file_name_col]}"

segmented_net_slope = compute_segment_slope(
    segmented_net,
    raster_path,
    crs_meter_epsg=2056,
    length_col="length",
    smooth_sigma=1.0,
    smooth_hops=2,
)
segmented_net_slope['pente'] = segmented_net_slope['slope_pct_abs_smoothed']
segmented_net_slope['pente_brute'] = segmented_net_slope['slope_pct_abs']
segmented_net_slope['filtered'] = segmented_net_slope['pente'].notna().astype(int)

print("\nDescriptive statistics (pente lissée):")
print(segmented_net_slope['pente'].describe())

segmented_net_slope.to_crs(target_crs)[[
    'segment_id',
    'geometry',
    'pente',
    'pente_brute',
    'elev_start_m',
    'elev_end_m',
    'elev_delta_m',
    'slope_pct_signed',
    'filtered',
]].head()

Data initialized
Processing attribute: pente

Descriptive statistics (pente lissée):
count    620047.000000
mean          5.901454
std           5.394804
min           0.000000
25%           2.635355
50%           4.454121
75%           7.553621
max         355.475541
Name: pente, dtype: float64


,segment_id,geometry,pente,pente_brute,elev_start_m,elev_end_m,elev_delta_m,slope_pct_signed,filtered
0,000000,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",4.918177,4.148971,4.143306,2.768295,-1.375011,-4.148971,1
1,000001,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",4.918177,4.148971,2.768295,4.143306,1.375011,4.148971,1
2,000002,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",4.115405,2.151501,0.412987,1.488738,1.075750,2.151501,1
3,000003,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",3.204459,2.858099,1.488738,2.917788,1.429050,2.858099,1
4,000004,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",2.665215,2.285849,2.917788,4.060712,1.142924,2.285849,1


In [9]:
save(
    save_filtered_attributes,
    row,
    segmented_net_slope.to_crs(target_crs)[[
        'segment_id',
        'geometry',
        'pente',
        'pente_brute',
        'elev_start_m',
        'elev_end_m',
        'elev_delta_m',
        'slope_pct_signed',
        'slope_pct_abs',
        'slope_pct_abs_smoothed',
        'filtered',
    ]],
    attribute,
 )

Geometries cleaned
Filtered data saved for attribute: pente in format: parquet
Filtered data saved for attribute: pente in format: csv
Filtered data saved for attribute: pente in format: gpkg


## Vitesse

In [5]:

import re
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

# --- Paramètres --------------------------------------------------------------
# >>> VITESSE VÉLO (Claude, 2026-11-24) : graphe et fichier selon le réseau ; le fichier marche n'est pas touché
# AVANT : edges_path = f"{input_file_path}/networkGG/walk_edges_graph.geojson"
if network == "bike":
    edges_path = f"{input_file_path}/networkGG/bike_edges_graph.parquet"
else:
    edges_path = f"{input_file_path}/networkGG/walk_edges_graph.geojson"
# <<< FIN VITESSE VÉLO
tache_urbaine_path = f"{input_file_path}/network_agreg/AGGLO_TACHE_URBAINE_2022.gpkg"
out_dir = Path(f"{input_file_path}/attributs/GG/vitesse")
basename = "vitesse_bike_edges" if network == "bike" else "vitesse_all_edges"   # >>> VITESSE VÉLO (2026-11-24)
operation_crs = "EPSG:2056"

# ============================================================================
# 1_1 · Vitesse par tronçon
#   1) base : tache urbaine 2022 dilatée de TACHE_BUFFER_M
#      -> 50 km/h (bâti) / 80 km/h (non bâti)
#   2) type de route (seulement sans maxspeed explicite) :
#        pedestrian, path, track, steps, corridor -> 1 km/h (pas de voiture)
#        living_street -> 20 km/h, residential -> 30 km/h, service -> 30 km/h, track -> 10 km/h
#   3) maxspeed OSM explicite et numérique -> écrase tout le reste
#   speed_kph = vitesse retenue ; vitesse_type = zone | osm_tag | type_inferred
# ============================================================================

TACHE_BUFFER_M = 200.0   # dilatation de la tache urbaine (0 = aucune)

SPEED_URBAN_KMH = 50.0
SPEED_RURAL_KMH = 80.0



MAXSPEED_COLS = ["maxspeed:forward", "maxspeed:backward", "maxspeed"]

# --- Chargement --------------------------------------------------------------
edges = gpd.read_parquet(edges_path) if str(edges_path).endswith(".parquet") else gpd.read_file(edges_path)   # >>> VITESSE VÉLO

tache_urbaine = gpd.read_file(tache_urbaine_path)
tache_urbaine["geometry"] = tache_urbaine.geometry.force_2d().make_valid()
tache_urbaine = tache_urbaine.to_crs(operation_crs)[["geometry"]]
if TACHE_BUFFER_M > 0:
    tache_urbaine["geometry"] = tache_urbaine.geometry.buffer(TACHE_BUFFER_M)


# --- Helpers -----------------------------------------------------------------
_NUMERIC_RE = re.compile(r"\d+(\.\d+)?")
_LISTLIKE = (list, tuple, set, np.ndarray)

def parse_explicit_maxspeed(v):
    """Nombre seul (ex. '50', 50) -> float. Tout le reste ('walk', 'CH:urban',
    '50 mph', '30;50', 'none'...) -> NaN."""
    if v is None or isinstance(v, _LISTLIKE):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v) if np.isfinite(v) and v > 0 else np.nan
    s = str(v).strip()
    if _NUMERIC_RE.fullmatch(s):
        x = float(s)
        return x if x > 0 else np.nan
    return np.nan


def normalize_highway(v):
    """Liste OSM (ex. ['tertiary', 'primary']) -> première valeur."""
    if isinstance(v, _LISTLIKE):
        v = list(v)
        return str(v[0]) if len(v) else np.nan
    if v is None or pd.isna(v):
        return np.nan
    return str(v)


NO_CAR_OWN_HW = {"pedestrian", "corridor"}   # l'arête est elle-même un espace sans voiture
NO_CAR_SPEED_KMH = 0.0                        # repéré par vitesse_type = "no_car"
TYPE_SPEED_KMH = {"living_street": 20.0, "residential": 30.0, "service": 30.0}
TRACK_SPEED_KMH = 10.0                        # track isolé : véhicules agricoles


def compute_speed(edges_gdf, tache_gdf):
    df = edges_gdf.copy()
    if "speed_ctx_kmh" not in df.columns:
        raise ValueError("speed_ctx_kmh absent : relancer 0_1 (cellule vitesse de la section 7).")

    # 1) Base : milieu du tronçon dans la tache urbaine (inchangé)
    mid = gpd.GeoDataFrame(
        geometry=df.to_crs(operation_crs).geometry.interpolate(0.5, normalized=True),
        index=df.index, crs=operation_crs,
    )
    in_tache = gpd.sjoin(mid, tache_gdf, how="inner", predicate="intersects").index.unique()
    df["in_tache_urbaine"] = df.index.isin(in_tache)

    speed = pd.Series(np.where(df["in_tache_urbaine"], SPEED_URBAN_KMH, SPEED_RURAL_KMH),
                      index=df.index, dtype="float64")
    vtype = pd.Series("zone", index=df.index, dtype="object")

    # Route de référence : l'arête elle-même si c'est un axe routier gardé,
    # sinon la chaussée adjacente trouvée dans 0_1 (<= 15 m)
    hw = df["highway"].apply(normalize_highway)
    adj_hw = df["adj_highway"].apply(normalize_highway)
    is_proxy = df["walk_class"].eq("sidewalk_proxy_road_axis")
    has_adj = df["adj_dist_m"].notna()
    ref_hw = hw.where(is_proxy, adj_hw)

    # 2) Type de la route de référence
    type_speed = ref_hw.map(TYPE_SPEED_KMH)
    m = type_speed.notna()
    speed[m] = type_speed[m]
    vtype[m] = "type_inferred"

    # 3) Vitesse explicite : tag propre (proxy) ou chaussée adjacente (trottoir, traversée)
    ctx = pd.to_numeric(df["speed_ctx_kmh"], errors="coerce")
    m = ctx.notna() & (ctx > 0)
    speed[m] = ctx[m]
    vtype[m & is_proxy] = "osm_tag"
    vtype[m & ~is_proxy] = "osm_tag_adjacent"

    # 4) Pas de voiture : espace piéton en soi, ou aucune chaussée à côté
    isolated = ~is_proxy & ~has_adj
    m_track = isolated & hw.eq("track")
    speed[m_track] = TRACK_SPEED_KMH
    vtype[m_track] = "type_inferred"

    m_nocar = hw.isin(NO_CAR_OWN_HW) | (isolated & ~hw.eq("track"))
    speed[m_nocar] = NO_CAR_SPEED_KMH
    vtype[m_nocar] = "no_car"

    df["highway_norm"] = hw
    df["ref_highway"] = ref_hw
    df["speed_kph"] = speed
    df["vitesse_type"] = vtype
    return df


def print_speed_summary(df):
    length_m = df.to_crs(operation_crs).geometry.length
    summary = pd.DataFrame({
        "n_troncons": df.groupby("vitesse_type").size(),
        "longueur_km": length_m.groupby(df["vitesse_type"]).sum() / 1000,
    })
    summary["part_troncons_%"] = 100 * summary["n_troncons"] / summary["n_troncons"].sum()
    summary["part_longueur_%"] = 100 * summary["longueur_km"] / summary["longueur_km"].sum()
    print("Répartition par type de vitesse :")
    print(summary.round(1).to_string())

    z = df["vitesse_type"] == "zone"
    print(f"\nZone : {int((z & df['in_tache_urbaine']).sum())} tronçons à {SPEED_URBAN_KMH:g} km/h, "
          f"{int((z & ~df['in_tache_urbaine']).sum())} à {SPEED_RURAL_KMH:g} km/h")

    # Contrôle : classes de route qui reçoivent la vitesse de zone
    print("\nClasses highway notées par la zone (top 10) :")
    print(df.loc[z, "highway_norm"].value_counts(dropna=False).head(10).to_string())


# >>> VITESSE VÉLO (Claude, 2026-11-24) ----------------------------------------
# Graphe vélo : la route de référence est l'arête elle-même (le cycliste roule dessus).
#   1) base : tache urbaine -> 50 / 80 km/h (comme la marche)
#   2) type de route (sans tag) : living_street 20, residential 30, service 30
#   3) tag de zone OSM (zone:maxspeed, maxspeed:type, maxspeed="CH:zone30"...) : zone30 -> 30, urban -> 50...
#   4) maxspeed numérique explicite (maxspeed, :forward, :backward ; on garde le plus élevé) -> écrase tout
#   5) track -> 10 km/h (véhicules agricoles) ; voies dessinées à part (cycleway, path, footway,
#      pedestrian, bridleway...) -> 0 km/h « pas de voiture »
BIKE_NO_CAR_HW = {"cycleway", "path", "footway", "pedestrian", "bridleway", "steps", "corridor"}
BIKE_SPEED_TAG_COLS = ["maxspeed", "maxspeed:forward", "maxspeed:backward"]
BIKE_ZONE_TAG_COLS = ["zone:maxspeed", "maxspeed:type", "maxspeed"]
_ZONE_RE = re.compile(r"zone\s*:?\s*(\d+)", re.IGNORECASE)
ZONE_WORD_KMH = {"living_street": 20.0, "walk": 5.0, "urban": 50.0, "rural": 80.0,
                 "trunk": 100.0, "motorway": 120.0}


def parse_zone_speed(v):
    """Tag de zone OSM -> km/h : 'CH:zone30' -> 30, 'FR:urban' -> 50, 'CH:rural' -> 80 ; sinon NaN."""
    if isinstance(v, _LISTLIKE):
        v = list(v)[0] if len(v) else None
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().lower()
    m = _ZONE_RE.search(s)
    if m:
        return float(m.group(1))
    for word, kmh in ZONE_WORD_KMH.items():
        if s == word or s.endswith(":" + word):
            return kmh
    return np.nan


def compute_speed_bike(edges_gdf, tache_gdf):
    df = edges_gdf.copy()
    mid = gpd.GeoDataFrame(
        geometry=df.to_crs(operation_crs).geometry.interpolate(0.5, normalized=True),
        index=df.index, crs=operation_crs,
    )
    in_tache = gpd.sjoin(mid, tache_gdf, how="inner", predicate="intersects").index.unique()
    df["in_tache_urbaine"] = df.index.isin(in_tache)

    speed = pd.Series(np.where(df["in_tache_urbaine"], SPEED_URBAN_KMH, SPEED_RURAL_KMH),
                      index=df.index, dtype="float64")
    vtype = pd.Series("zone", index=df.index, dtype="object")
    hw = df["highway"].apply(normalize_highway)

    # 2) type de route
    type_speed = hw.map(TYPE_SPEED_KMH)
    m = type_speed.notna()
    speed[m] = type_speed[m]
    vtype[m] = "type_inferred"

    # 3) tag de zone
    zone = pd.Series(np.nan, index=df.index)
    for c in BIKE_ZONE_TAG_COLS:
        if c in df.columns:
            zone = zone.fillna(df[c].map(parse_zone_speed))
    m = zone.notna()
    speed[m] = zone[m]
    vtype[m] = "osm_zone_tag"

    # 4) maxspeed numérique (le plus élevé des sens)
    cols = [c for c in BIKE_SPEED_TAG_COLS if c in df.columns]
    explicit = df[cols].apply(lambda col: col.map(parse_explicit_maxspeed)).max(axis=1) if cols else pd.Series(np.nan, index=df.index)
    m = explicit.notna()
    speed[m] = explicit[m]
    vtype[m] = "osm_tag"

    # 5) track et voies sans voiture
    m = hw.eq("track")
    speed[m] = TRACK_SPEED_KMH
    vtype[m] = "type_inferred"
    m = hw.isin(BIKE_NO_CAR_HW)
    speed[m] = NO_CAR_SPEED_KMH
    vtype[m] = "no_car"

    df["highway_norm"] = hw
    df["speed_kph"] = speed
    df["vitesse_type"] = vtype
    return df
# <<< FIN VITESSE VÉLO -----------------------------------------------------------


def export_speed_layer(df, out_dir, basename):
    out_dir.mkdir(parents=True, exist_ok=True)
    keep = [c for c in ["edge_id", "u", "v", "key", "osm_id", "id", "element", "name",
                        "highway", "highway_norm", "walk_class", "ref_highway",
                        "adj_osm_id", "adj_highway", "adj_dist_m", "speed_ctx_kmh",
                        "maxspeed", "zone:maxspeed", "maxspeed:type", "infra_cyclable",   # >>> VITESSE VÉLO
                        "in_tache_urbaine", "speed_kph", "vitesse_type",
                        "length", "len_m", "geometry"] if c in df.columns]
    out = df[keep].copy()
    # listes OSM éventuelles -> texte, pour l'export
    for c in out.columns.drop("geometry"):
        if out[c].map(lambda x: isinstance(x, _LISTLIKE)).any():
            out[c] = out[c].map(lambda x: ";".join(map(str, x)) if isinstance(x, _LISTLIKE) else x)

    out.to_file(out_dir / f"{basename}.geojson", driver="GeoJSON")
    out.to_parquet(out_dir / f"{basename}.parquet", index=False)
    print("\nExported:", out_dir / f"{basename}.geojson")
    print("Exported:", out_dir / f"{basename}.parquet")
    print("Missing speed_kph:", int(out["speed_kph"].isna().sum()))
    return out


# --- Exécution ---------------------------------------------------------------
# >>> VITESSE VÉLO : AVANT speed_edges = compute_speed(edges, tache_urbaine)
speed_edges = compute_speed_bike(edges, tache_urbaine) if network == "bike" else compute_speed(edges, tache_urbaine)
print_speed_summary(speed_edges)
speed_gdf = export_speed_layer(speed_edges, out_dir, basename)

/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/venv/lib/python3.11/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


Répartition par type de vitesse :
               n_troncons  longueur_km  part_troncons_%  part_longueur_%
vitesse_type                                                            
no_car              42426       5875.8             20.8             23.1
osm_tag             65001       5553.4             31.9             21.9
osm_zone_tag          877         91.4              0.4              0.4
type_inferred       71407      10498.4             35.0             41.4
zone                24162       3364.4             11.9             13.3

Zone : 21399 tronçons à 50 km/h, 2763 à 80 km/h

Classes highway notées par la zone (top 10) :
highway_norm
unclassified      15432
tertiary           5314
secondary          1931
primary            1265
primary_link        102
tertiary_link        70
secondary_link       48

Exported: ../../Data/input/attributs/GG/vitesse/vitesse_bike_edges.geojson
Exported: ../../Data/input/attributs/GG/vitesse/vitesse_bike_edges.parquet
Missing speed_kph: 0


## Traversées : contrôle

Cette section **n'alimente plus l'indice**. Le score `traversees` est produit par la chaîne standard :
- **2_1** : points OSM filtrés → couches `n_traversees` et `n_traversees_feux` (règles dans `crossing_filters.py`) ;
- **2_2, boucle V3** : comptage par segment avec `how = count_rectangle` (rectangle de demi-largeur `buffer_size_GG`, segment prolongé de `extend_m_GG`) ;
- **2_2, section « Attributs dérivés »** : `traversees = 1 − besoin × (1 − offre)` (`derived_attributes.py`).

Ici on recalcule la même chose avec l'implémentation de référence (sjoin `within`) et on exporte dans `Data/output/{network}/{territory}/traversees_control/` :
- `traversees_control_segments.parquet` : table par segment (vitesse, comptages, besoin, offre, score), comparée à `step2_features` en fin de 2_2 ;
- `traversees_segments.gpkg` : réseau avec ces colonnes (lignes) ;
- `traversees_rectangles.gpkg` : rectangles de comptage (si `EXPORT_RECTANGLES`) ;
- `traversees_crossings_retenues.gpkg` : traversées retenues, avec `crossing_tag` et `is_signal`.

Les paramètres du rectangle sont lus dans l'Excel (ligne `n_traversees`). À lancer après la cellule Vitesse.

In [11]:
# ============================================================================
# 1_1 · Traversées : CONTRÔLE (n'alimente pas l'indice)
#   Recalcule, avec l'implémentation de référence (sjoin "within"), les comptages
#   que la boucle V3 de 2_2 produit avec how="count_rectangle", puis le score dérivé.
#   Mêmes règles de sélection (crossing_filters.py), même rectangle
#   (feature_to_network.segment_rectangles), même score (derived_attributes.py).
# ============================================================================
import importlib
import numpy as np
import pandas as pd
import geopandas as gpd

import crossing_filters, derived_attributes, feature_to_network
for _m in (crossing_filters, derived_attributes, feature_to_network):
    importlib.reload(_m)
from crossing_filters import load_crossings, filter_crossings
from feature_to_network import segment_rectangles
from derived_attributes import crossing_score_table

# --- Entrées et sorties -------------------------------------------------------
crossings_path = f"{input_file_path}/networkGG/walk_crossing_nodes.parquet"
speed_path     = f"{input_file_path}/attributs/GG/vitesse/vitesse_all_edges.parquet"
control_dir    = f"../../Data/output/{network}/{territory}/traversees_control"

EXPORT_RECTANGLES = True   # gpkg des rectangles : ~1 polygone par segment, lourd

# Paramètres du rectangle lus dans l'Excel (ligne n_traversees), comme en 2_2.
# On relit l'Excel complet : `attributs_info` (cellule « Import des attributs »)
# exclut les lignes include_in_index = False, dont n_traversees.
_ai = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}_urbain.xlsx",
                    sheet_name="attributs_info")
_row = _ai.loc[_ai["attribute"] == "n_traversees"].iloc[0]
CROSSING_BUFFER_M = float(_row[f"buffer_size_{territory}"])
_ext = _row.get(f"extend_m_{territory}", np.nan)
CROSSING_EXTEND_M = float(_ext) if pd.notna(_ext) else 0.0
print(f"Rectangle : demi-largeur {CROSSING_BUFFER_M} m, prolongement {CROSSING_EXTEND_M} m à chaque bout")


def count_crossings(segments, crossings, half_width, extend):
    """Référence : traversées à l'intérieur (within) du rectangle de chaque segment."""
    rect = gpd.GeoDataFrame(geometry=segment_rectangles(segments, half_width, extend))
    hits = gpd.sjoin(crossings, rect, how="inner", predicate="within")
    n_all = hits.groupby("index_right").size()
    n_sig = hits.loc[hits["is_signal"]].groupby("index_right").size()
    res = pd.DataFrame(index=segments.index)
    res["n_traversees"] = n_all.reindex(res.index, fill_value=0).astype(int)
    res["n_traversees_feux"] = n_sig.reindex(res.index, fill_value=0).astype(int)
    return res, rect


# --- Exécution ----------------------------------------------------------------
# 1) Segments + vitesse propre de l'arête (jointure sur edge_id)
seg = segmented_net.to_crs(operation_crs)[["segment_id", "edge_id", "geometry"]].copy()
speed = (pd.read_parquet(speed_path, columns=["edge_id", "speed_kph"])
           .drop_duplicates("edge_id"))
seg = seg.merge(speed, on="edge_id", how="left")
print(f"Segments sans vitesse : {int(seg['speed_kph'].isna().sum()):,} / {len(seg):,}")

# 2) Traversées retenues (règles partagées) et comptage dans les rectangles
crossings = filter_crossings(load_crossings(crossings_path, operation_crs))   # drop=True
counts, rectangles = count_crossings(seg, crossings, CROSSING_BUFFER_M, CROSSING_EXTEND_M)
seg = seg.join(counts)

# 3) Score (même fonction que la section « Attributs dérivés » de 2_2)
tab = crossing_score_table(seg["speed_kph"], seg["n_traversees"], seg["n_traversees_feux"])
seg = seg.join(tab.rename(columns={"besoin": "besoin_traversee",
                                   "offre": "offre_traversee",
                                   "score": "traversees"}))

print("\nClasses de vitesse (segments) :")
print(seg["vitesse_classe"].value_counts(dropna=False).to_string())
print(f"\nSegments avec ≥ 1 traversée : {int((seg['n_traversees'] > 0).sum()):,}")
print("\nScore moyen par classe :")
print(seg.groupby("vitesse_classe")[["traversees", "offre_traversee"]].mean().round(3).to_string())

traversees_control = seg
traversees_control.head()

FileNotFoundError: [Errno 2] No such file or directory: '../../Data/input/attributs/attributs_info_bike_urbain.xlsx'

In [ ]:
# Export du contrôle traversées -> Data/output/{network}/{territory}/traversees_control/
os.makedirs(control_dir, exist_ok=True)

ctrl_cols = ["segment_id", "edge_id", "speed_kph", "vitesse_classe",
             "n_traversees", "n_traversees_feux", "besoin_traversee", "offre_traversee", "traversees"]

# Table par segment (sans géométrie) : comparée à step2_features en fin de 2_2
pd.DataFrame(traversees_control[ctrl_cols]).to_parquet(
    f"{control_dir}/traversees_control_segments.parquet", index=False)

# Couches QGIS (EPSG:2056, comme le calcul)
traversees_control[ctrl_cols + ["geometry"]].to_file(
    f"{control_dir}/traversees_segments.gpkg", driver="GPKG")
crossings.to_file(f"{control_dir}/traversees_crossings_retenues.gpkg", driver="GPKG")

if EXPORT_RECTANGLES:
    rect_out = gpd.GeoDataFrame(traversees_control[ctrl_cols].copy(),
                                geometry=rectangles.geometry.values, crs=operation_crs)
    rect_out.to_file(f"{control_dir}/traversees_rectangles.gpkg", driver="GPKG")

print(f"Contrôle traversées exporté dans {control_dir}")
print(sorted(os.listdir(control_dir)))

## Merge files

In [12]:
#Helper

territories = ['GE', 'FR', 'VD']

# ── Chargement générique avec gestion des valeurs manquantes ──
def load_territory(attributs_info, attribute, territory, input_file_path, target_crs):
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    source_path = row.get(f'source_path_{territory}')
    file_name   = row.get(f'file_name_{territory}')

    # Cas manquants : cellule vide, NaN, ou chemin inexistant
    if pd.isna(source_path) or pd.isna(file_name):
        print(f"[{territory}] Pas de fichier défini: territoire ignoré.")
        return None

    full_path = f"{input_file_path}/{source_path}/{file_name}"

    if not os.path.exists(full_path):
        print(f"[{territory}] Fichier introuvable : {full_path} territoire ignoré.")
        return None

    gdf = gpd.read_file(full_path)
    return gdf.to_crs(target_crs)

### Sens inverse

In [13]:
# ── CAHNGE HERE ──────────────────────────────────────────────
attribute = 'sens_inverse'
#──────────────────────────────────────────────

# ── Chargement de tous les territoires ────────────────────────
gdfs = {}
for territory in territories:
    result = load_territory(
        attributs_info, attribute, territory, input_file_path, target_crs
    )
    if result is not None:
        result['territory'] = territory
        gdfs[territory] = result

# ── Merge uniquement si au moins un territoire chargé ─────────
if gdfs:
    attribute_merged = gpd.GeoDataFrame(
        pd.concat(gdfs.values(), ignore_index=True),
        crs=target_crs
    )
    print(f"Merge réussi : {len(attribute_merged)} lignes — territoires : {list(gdfs.keys())}")
else:
    print(f"Aucun fichier trouvé pour l'attribut '{attribute}'.")
    attribute_merged = None


#Save merged file into input/attributs/GG/filename

if attribute_merged is not None:
    output_dir  = os.path.join(input_file_path, 'attributs', 'GG', attribute)
    output_file = os.path.join(output_dir, f"{attribute}.gpkg")
    
    os.makedirs(output_dir, exist_ok=True)
    attribute_merged.to_file(output_file, driver='GPKG')
    
    print(f"Fichier sauvegardé : {output_file}")
else:
    print(f"Aucun fichier à sauvegarder pour '{attribute}'.")

[FR] Pas de fichier défini: territoire ignoré.
[VD] Pas de fichier défini: territoire ignoré.
Merge réussi : 6727 lignes — territoires : ['GE']
Fichier sauvegardé : ../../Data/input/attributs/GG/sens_inverse/sens_inverse.gpkg


### Tourner à droite ###

In [14]:
# ── CHANGE HERE ──────────────────────────────────────────────
attribute = 'tourner_droite'
#──────────────────────────────────────────────

# ── Chargement de tous les territoires ────────────────────────
gdfs = {}
for territory in territories:
    result = load_territory(
        attributs_info, attribute, territory, input_file_path, target_crs
    )
    if result is not None:
        result['territory'] = territory
        gdfs[territory] = result

# ── Merge uniquement si au moins un territoire chargé ─────────
if gdfs:
    attribute_merged = gpd.GeoDataFrame(
        pd.concat(gdfs.values(), ignore_index=True),
        crs=target_crs
    )
    print(f"Merge réussi : {len(attribute_merged)} lignes — territoires : {list(gdfs.keys())}")
else:
    print(f"Aucun fichier trouvé pour l'attribut '{attribute}'.")
    attribute_merged = None


#Save merged file into input/attributs/GG/filename

if attribute_merged is not None:
    output_dir  = os.path.join(input_file_path, 'attributs', 'GG', attribute)
    output_file = os.path.join(output_dir, f"{attribute}.gpkg")
    
    os.makedirs(output_dir, exist_ok=True)
    attribute_merged.to_file(output_file, driver='GPKG')
    
    print(f"Fichier sauvegardé : {output_file}")
else:
    print(f"Aucun fichier à sauvegarder pour '{attribute}'.")

[FR] Pas de fichier défini: territoire ignoré.
[VD] Pas de fichier défini: territoire ignoré.
Merge réussi : 11726 lignes — territoires : ['GE']
Fichier sauvegardé : ../../Data/input/attributs/GG/tourner_droite/tourner_droite.gpkg
